# Milestone 2 – Adapter Training

| Required evidence | Section |
|---|---|
| Trainable parameter summary | 2 |
| Loss history | 3 |
| Adapter files saved separately from the base model | 4 |
| A command that reloads the adapter for inference | 5 |

The logic is in `src/prompts.py`, `src/training.py`, and `src/generation.py`. This notebook runs it and shows the results.

Train from the terminal (about 5 minutes on a laptop CPU):

```bash
uv run python -m src.training
```

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

# Run from the project root so data/, src/ and artifacts/ resolve.
if Path.cwd().name == "notebooks":
    os.chdir("..")

from src.training import SUMMARY_FILENAME, TrainingConfig, train_adapter

print("Working directory:", Path.cwd())

## 1. Training configuration and run

Settings come from `.env`. The LoRA settings (`r`, `alpha`, `dropout`, and the target modules) are set in `TrainingConfig`.
The training data is validated first, and training stops if validation finds errors.

Set `RETRAIN = True` to run the optimization loop again. Otherwise the summary saved by the last run is loaded.

In [ ]:
RETRAIN = False

config = TrainingConfig()
summary_path = config.adapter_path / SUMMARY_FILENAME
if RETRAIN or not summary_path.exists():
    summary = train_adapter(config)
else:
    summary = json.loads(summary_path.read_text(encoding="utf-8"))

for key in [
    "base_model", "base_model_revision", "prompt_format",
    "target_modules", "lora_rank", "lora_alpha", "lora_dropout",
    "max_sequence_length", "epochs", "learning_rate", "seed",
    "training_examples", "training_sha256", "device", "training_seconds", "adapter_path",
]:
    print(f"{key:<22} {summary[key]}")

## 2. Trainable parameter summary

LoRA freezes the base model and adds two small matrices to each target projection: `A` (r × in) and `B` (out × r).
Only these matrices are trained.

In [ ]:
trainable, total = summary["trainable_parameters"], summary["total_parameters"]
print(f"Trainable (LoRA) parameters: {trainable:>12,}")
print(f"Frozen base parameters:      {total - trainable:>12,}")
print(f"Total parameters:            {total:>12,}")
print(f"Trainable share:             {summary['trainable_percentage']:>11.4f}%")

In [ ]:
# Check the count: load the base model with the adapter and list the LoRA tensors.
from src.generation import load_model

adapted_model, tokenizer = load_model(config.adapter_path)
adapted_model.print_trainable_parameters()  # 0 trainable: the adapter loads frozen for inference

lora_tensors = {name: p for name, p in adapted_model.named_parameters() if "lora_" in name}
lora_total = sum(p.numel() for p in lora_tensors.values())
print(f"\n{len(lora_tensors)} LoRA tensors, {lora_total:,} parameters "
      f"(matches training summary: {lora_total == trainable})")
print("\nLayer 0 example:")
for name, p in lora_tensors.items():
    if ".layers.0." in name:
        print(f"  {name.split('layers.0.')[1]:<42} {tuple(p.shape)}")

## 3. Loss history

Mean training loss per epoch over the answer tokens only, because the prompt tokens are masked with `-100`.
Each epoch shuffles the examples with a fixed seed and uses batch size 1.

In [ ]:
epoch_losses = summary["epoch_losses"]
step_losses = summary["step_losses"]
steps_per_epoch = len(step_losses) // len(epoch_losses)

print(f"{'epoch':>5}  {'mean':>6}  {'min':>6}  {'max':>6}")
for epoch, mean in enumerate(epoch_losses, start=1):
    steps = step_losses[(epoch - 1) * steps_per_epoch : epoch * steps_per_epoch]
    bar = "#" * round(mean * 30)
    print(f"{epoch:>5}  {mean:>6.3f}  {min(steps):>6.3f}  {max(steps):>6.3f}  {bar}")

print(f"\nLoss went from {epoch_losses[0]:.3f} to {epoch_losses[-1]:.3f} "
      f"({100 * (1 - epoch_losses[-1] / epoch_losses[0]):.0f}% lower) over {len(step_losses)} steps.")

A falling training loss only shows that the optimizer fit the 16 training examples.
It does not show that the adapter answers held-out questions better. Milestone 4 tests that on `data/evaluation/cases.json`.
With 16 examples and 8 epochs, a loss that keeps falling may also be overfitting.

## 4. Adapter files are saved separately from the base model

`PeftModel.save_pretrained` writes only the LoRA weights and their config.
The base model stays in the Hugging Face cache, and the adapter refers to it by name.

In [ ]:
def size_mb(path: Path) -> float:
    files = [path] if path.is_file() else [p for p in path.rglob("*") if p.is_file()]
    return sum(p.stat().st_size for p in files) / 1024**2

print(f"Adapter folder: {config.adapter_path}")
for path in sorted(config.adapter_path.iterdir()):
    print(f"  {path.name:<28} {path.stat().st_size / 1024:>8.1f} KB")

from huggingface_hub import snapshot_download

base_dir = Path(snapshot_download(summary["base_model"], revision=summary["base_model_revision"]))
adapter_mb, base_mb = size_mb(config.adapter_path), size_mb(base_dir)
print(f"\nBase model snapshot: {base_dir}")
print(f"Adapter size: {adapter_mb:.1f} MB   Base model size: {base_mb:.1f} MB   "
      f"({100 * adapter_mb / base_mb:.1f}% of the base)")

In [ ]:
adapter_config = json.loads((config.adapter_path / "adapter_config.json").read_text(encoding="utf-8"))
for key in ["base_model_name_or_path", "revision", "peft_type", "task_type",
            "target_modules", "r", "lora_alpha", "lora_dropout"]:
    print(f"{key:<24} {adapter_config.get(key)}")

## 5. Reload the adapter for inference

The command loads the pinned base model, attaches the saved adapter with `PeftModel.from_pretrained`, retrieves evidence from the policies, and generates a reply:

```bash
uv run python -m src.generation --adapter artifacts/support-lora-adapter \
    --question "A customer forgot their password. What should support do?"
```

Leave out `--adapter` to get the unchanged base model for comparison.

In [ ]:
QUESTION = "A customer forgot their password. What should support do?"

for adapter_args in ([], ["--adapter", str(config.adapter_path)]):
    command = [sys.executable, "-m", "src.generation", *adapter_args, "--question", QUESTION]
    print("$ python -m src.generation", " ".join(adapter_args), "--question ...")
    result = subprocess.run(command, capture_output=True, text=True, encoding="utf-8")
    print("\n".join(line for line in result.stdout.splitlines()
                    if line.startswith(("Retrieved", "Model", "Answer"))) or result.stderr[-2000:])
    print()

In [ ]:
# Confirm the trained weights were loaded: a fresh LoRA B matrix starts at zero.
lora_b = next(p for name, p in adapted_model.named_parameters() if "lora_B" in name)
print("First lora_B matrix, mean |value|:", float(lora_b.abs().mean()))
print("Non-zero, so trained weights were loaded:", bool(lora_b.abs().sum() > 0))

**Observation (one question, not a benchmark).** The base model writes a long, generic reply with no citation.
The adapted model uses the trained style: a short, action-first reply ending in `[account-security]`.
But its content can be wrong. In this run it told support to *ask for the password*, which the policy forbids.
So the adapter can improve style while hurting faithfulness. Milestone 3 must check citations, and Milestone 4 must measure faithfulness separately from style.